<a href="https://colab.research.google.com/github/SalvaPiol45Tech/Multimodel-/blob/main/MiniLLaVA_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# Install libraries
import subprocess
import sys

print("📦 Installing libraries...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "transformers"])
print("✅ Done!\n")

# Imports
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import CLIPProcessor, CLIPModel, GPT2Tokenizer, GPT2LMHeadModel
from PIL import Image
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Device setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
class ProjectionLayer(nn.Module):
    """
   CLIP image space (768) لـ GPT-2 text space (768)


    768 (CLIP) → 1024 (hidden) → 768 (GPT-2)
     non-linearity (GELU)
    """

    def __init__(self, input_dim=768, hidden_dim=1024, output_dim=768):
        super().__init__()

        # Layer 1: expand
        self.fc1 = nn.Linear(input_dim, hidden_dim)

        # Activation function
        self.gelu = nn.GELU()

        # Layer 2: project back
        self.fc2 = nn.Linear(hidden_dim, output_dim)

        # Normalization
        self.layer_norm = nn.LayerNorm(output_dim)

    def forward(self, x):
        """
        x: (batch_size, 768)
        return: (batch_size, 768)
        """
        x = self.fc1(x)           # 768 → 1024
        x = self.gelu(x)          # Non-linearity
        x = self.fc2(x)           # 1024 → 768
        x = self.layer_norm(x)    # Stabilize
        return x

# Test الـ layer
print("✅ ProjectionLayer defined!")
test_proj = ProjectionLayer()
test_input = torch.randn(2, 768)
test_output = test_proj(test_input)
print(f"Input shape: {test_input.shape}")
print(f"Output shape: {test_output.shape}")

In [ ]:
class MiniLLaVA(nn.Module):
    """
     Vision-Language Model:
    Image → CLIP (frozen) → Projection (trainable) → GPT-2 (frozen) → Caption
    """

    def __init__(self, freeze_clip=True, freeze_gpt2=True):
        super().__init__()

        # Load CLIP encoder
        print("🔄 Loading CLIP...")
        self.clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
        self.clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

        if freeze_clip:
            for param in self.clip_model.parameters():
                param.requires_grad = False
        print("✅ CLIP loaded and frozen")

        # Load GPT-2 decoder
        print("🔄 Loading GPT-2...")
        self.gpt2_model = GPT2LMHeadModel.from_pretrained("gpt2").to(device)
        self.tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
        self.tokenizer.pad_token = self.tokenizer.eos_token

        if freeze_gpt2:
            for param in self.gpt2_model.parameters():
                param.requires_grad = False
        print("✅ GPT-2 loaded and frozen")

        # Build Projection Layer (trainable)
        print("🔄 Building Projection Layer...")
        self.projection = ProjectionLayer(512, 1024, 512).to(device)
        print("✅ Projection Layer built (trainable)\n")

    def encode_image(self, images):
        """
        image features  CLIP
        images: list  PIL Image objects
        return: (batch_size, 768)
        """
        with torch.no_grad():
            if not isinstance(images, list):
                images = [images]

            inputs = self.clip_processor(images=images, return_tensors="pt").to(device)
            vision_outputs = self.clip_model.vision_model(**inputs)
            image_features = self.clip_model.visual_projection(vision_outputs.pooler_output)

        return image_features

    def forward(self, images, input_ids, attention_mask=None):
        """
        Forward pass:
        1. Encode image with CLIP
        2. Project to text space
        3. Combine with text tokens
        4. Forward through GPT-2
        """

        # Step 1: Get image features
        image_features = self.encode_image(images)  # (batch, 768)

        # Step 2: Project to text space
        projected_features = self.projection(image_features)  # (batch, 768)

        # Step 3: Get text embeddings
        text_embeddings = self.gpt2_model.transformer.wte(input_ids)  # (batch, seq_len, 768)

        # Step 4: Combine [visual_token] + [text_tokens]
        batch_size = projected_features.shape[0]
        visual_tokens = projected_features.unsqueeze(1)  # (batch, 1, 768)
        combined_embeddings = torch.cat([visual_tokens, text_embeddings], dim=1)

        # Step 5: Create attention mask
        if attention_mask is not None:
            visual_attention = torch.ones(batch_size, 1, device=attention_mask.device)
            attention_mask = torch.cat([visual_attention, attention_mask], dim=1)

        # Step 6: Forward through GPT-2
        outputs = self.gpt2_model(
            inputs_embeds=combined_embeddings,
            attention_mask=attention_mask,
            return_dict=True
        )

        return outputs.logits

    def count_trainable_params(self):
        """ parameters """
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

# Initialize model
print("="*60)
print(" Initializing MiniLLaVA Model")
print("="*60)
model = MiniLLaVA(freeze_clip=True, freeze_gpt2=True)

# Print stats
total_params = sum(p.numel() for p in model.parameters())
trainable_params = model.count_trainable_params()

print(f"\n📊 Model Statistics:")
print(f"   Total parameters: {total_params:,}")
print(f"   Trainable parameters: {trainable_params:,}")
print(f"   Frozen: CLIP + GPT-2")
print(f"   Training: Projection Layer only")

In [ ]:
class ImageCaptionDataset(Dataset):
    """Dataset لـ medical images و captions"""

    def __init__(self, images, captions, tokenizer, max_length=128):
        self.images = images
        self.captions = captions
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx]
        if isinstance(image, str):
            image = Image.open(image).convert('RGB')

        caption = self.captions[idx]

        tokens = self.tokenizer(
            caption,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        return {
            'image': image,
            'input_ids': tokens['input_ids'].squeeze(),
            'attention_mask': tokens['attention_mask'].squeeze()
        }

print("✅ Dataset class defined!\n")

# Custom collate function
def collate_fn(batch):
    images = [item['image'] for item in batch]
    input_ids = torch.stack([item['input_ids'] for item in batch])
    attention_mask = torch.stack([item['attention_mask'] for item in batch])

    return {
        'image': images,
        'input_ids': input_ids,
        'attention_mask': attention_mask
    }

# Create dummy dataset
print("📁 Creating dummy medical dataset...")

dummy_images = [
    Image.new('RGB', (224, 224), color=(80, 80, 80)),
    Image.new('RGB', (224, 224), color=(100, 100, 100)),
    Image.new('RGB', (224, 224), color=(120, 120, 120)),
    Image.new('RGB', (224, 224), color=(140, 140, 140)),
]

dummy_captions = [
    "Normal chest X-ray with clear lungs",
    "Pneumonia detected in lower left lobe",
    "Mild pleural effusion on right side",
    "Clear bilateral lungs without disease"
]

dataset = ImageCaptionDataset(
    images=dummy_images,
    captions=dummy_captions,
    tokenizer=model.tokenizer,
    max_length=50
)

print(f"Dataset size: {len(dataset)}\n")

# Create DataLoader
print("📊 Creating DataLoader...")
train_loader = DataLoader(
    dataset,
    batch_size=2,
    shuffle=True,
    collate_fn=collate_fn
)

print(f"DataLoader batches: {len(train_loader)}")

# Test one batch
print("\n🧪 Testing batch...")
batch = next(iter(train_loader))
print(f"Images: {len(batch['image'])}")
print(f"input_ids shape: {batch['input_ids'].shape}")
print(f"attention_mask shape: {batch['attention_mask'].shape}")
print("✅ Dataset ready!")

In [ ]:
# Fix: Simple projection layer
class SimpleProjection(nn.Module):
    def __init__(self):
        super().__init__()
        # 512 (CLIP) → 768 (GPT-2)
        self.linear = nn.Linear(512, 768)

    def forward(self, x):
        return self.linear(x)

print("✅ SimpleProjection defined")

# Rebuild model with simple projection
class MiniLLaVA_Simple(nn.Module):
    def __init__(self):
        super().__init__()

        print("🔄 Loading CLIP...")
        self.clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
        self.clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
        for param in self.clip_model.parameters():
            param.requires_grad = False

        print("🔄 Loading GPT-2...")
        self.gpt2_model = GPT2LMHeadModel.from_pretrained("gpt2").to(device)
        self.tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
        self.tokenizer.pad_token = self.tokenizer.eos_token
        for param in self.gpt2_model.parameters():
            param.requires_grad = False

        print("🔄 Building Projection...")
        self.projection = SimpleProjection().to(device)
        print("✅ Model ready\n")

    def encode_image(self, images):
        with torch.no_grad():
            if not isinstance(images, list):
                images = [images]
            inputs = self.clip_processor(images=images, return_tensors="pt").to(device)
            vision_outputs = self.clip_model.vision_model(**inputs)
            image_features = self.clip_model.visual_projection(vision_outputs.pooler_output)
        return image_features

    def forward(self, images, input_ids, attention_mask=None):
        # Get image features (512)
        image_features = self.encode_image(images)

        # Project to 768
        projected = self.projection(image_features)

        # Get text embeddings (768)
        text_embeddings = self.gpt2_model.transformer.wte(input_ids)

        # Combine
        batch_size = projected.shape[0]
        visual_tokens = projected.unsqueeze(1)  # (batch, 1, 768)
        combined = torch.cat([visual_tokens, text_embeddings], dim=1)

        # Attention mask
        if attention_mask is not None:
            visual_att = torch.ones(batch_size, 1, device=attention_mask.device)
            attention_mask = torch.cat([visual_att, attention_mask], dim=1)

        # Forward GPT-2
        outputs = self.gpt2_model(
            inputs_embeds=combined,
            attention_mask=attention_mask,
            return_dict=True
        )

        return outputs.logits

    def count_trainable_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

# Initialize
print("="*60)
print("Reinitializing model...")
print("="*60)
model = MiniLLaVA_Simple()
print(f"Trainable params: {model.count_trainable_params():,}\n")

In [ ]:
def train_model_fixed(model, train_loader, num_epochs=3, learning_rate=1e-4):
    """
     MiniLLaVA
    """

    print("\n" + "="*70)
    print("🎓 TRAINING MINILLAVA")
    print("="*70)
    print(f"Trainable params: {model.count_trainable_params():,}")
    print(f"Epochs: {num_epochs}")
    print(f"Learning rate: {learning_rate}")
    print(f"Device: {device}")
    print("="*70 + "\n")

    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=learning_rate
    )

    loss_fn = nn.CrossEntropyLoss()
    model.train()

    for epoch in range(num_epochs):
        total_loss = 0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")

        for batch in pbar:
            images = batch['image']
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)

            # Forward pass
            logits = model(images, input_ids, attention_mask)
            # logits: (batch, seq_len+1, vocab_size)
            # visual token

            # Remove visual token logits, keep only text logits
            # logits[..., 1:, :] → (batch, seq_len, vocab_size)
            text_logits = logits[..., 1:, :]

            # Shift for next token prediction
            shift_logits = text_logits[..., :-1, :].contiguous()
            shift_labels = input_ids[..., 1:].contiguous()

            # Calculate loss
            loss = loss_fn(
                shift_logits.view(-1, shift_logits.shape[-1]),
                shift_labels.view(-1)
            )

            # Backward
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            total_loss += loss.item()
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})

        avg_loss = total_loss / len(train_loader)
        print(f"✅ Epoch {epoch+1} - Avg Loss: {avg_loss:.4f}\n")

    return model

print("✅ Fixed training function defined!\n")

# Train
print("🚀 Starting training...\n")
model = train_model_fixed(
    model=model,
    train_loader=train_loader,
    num_epochs=3,
    learning_rate=1e-4
)

print("✅ Training completed!")

In [ ]:
def generate_caption(model, image, max_tokens=50, temperature=1.0):
    """
     caption
    """
    model.eval()

    with torch.no_grad():
        # Encode image
        image_features = model.encode_image(image)
        projected = model.projection(image_features)  # (1, 768)

        # Start generation
        input_ids = torch.tensor([[model.tokenizer.bos_token_id]]).to(device)

        # Generate tokens one by one
        for _ in range(max_tokens):
            # Get text embeddings
            text_emb = model.gpt2_model.transformer.wte(input_ids)

            # Combine with visual token
            visual_tokens = projected.unsqueeze(1)
            combined = torch.cat([visual_tokens, text_emb], dim=1)

            # Forward
            outputs = model.gpt2_model(inputs_embeds=combined)
            logits = outputs.logits[:, -1, :] / temperature

            # Get next token
            next_token = torch.argmax(logits, dim=-1).unsqueeze(-1)

            # Stop if EOS
            if next_token.item() == model.tokenizer.eos_token_id:
                break

            input_ids = torch.cat([input_ids, next_token], dim=1)

        # Decode
        caption = model.tokenizer.decode(
            input_ids[0],
            skip_special_tokens=True
        )

    return caption

print(" Inference function defined!\n")

# Test on multiple images
print("="*70)
print(" TESTING INFERENCE")
print("="*70 + "\n")

test_images = [
    Image.new('RGB', (224, 224), color=(50, 50, 50)),
    Image.new('RGB', (224, 224), color=(100, 100, 100)),
    Image.new('RGB', (224, 224), color=(150, 150, 150)),
]

for i, test_img in enumerate(test_images):
    caption = generate_caption(model, test_img, max_tokens=40)
    print(f"Test Image {i+1}:")
    print(f"Generated: {caption}\n")

print("="*70)
print(" MiniLLaVA TRAINING & INFERENCE COMPLETE!")
print("="*70)

print("\n Summary:")
print(f"    Model: CLIP + Projection + GPT-2")
print(f"    Trainable params: {model.count_trainable_params():,}")
print(f"    Training: 3 epochs completed")
print(f"    Inference: Captions generated")

print("\n🎯 Next steps for real data:")
print("   1. Download CheXpert from Kaggle")
print("   2. Load real X-ray images")
print("   3. Create medical captions")
print("   4. Train on full dataset (more epochs)")
print("   5. Evaluate on test set")
print("   6. Save model: torch.save(model.state_dict(), 'minillava.pt')")
print("\n" + "="*70)

In [ ]:
# Save model
print("💾 Saving model...")
torch.save(model.state_dict(), 'minillava_model.pt')
print("✅ Model saved as 'minillava_model.pt'\n")

# Load model later
print("📝 To load later:")
print("   model = MiniLLaVA_Simple()")
print("   model.load_state_dict(torch.load('minillava_model.pt'))")